# Literature Review and Data Collection Plan

The 15-minute city has become a compact way to describe a much older planning ambition: residents should be able to satisfy daily needs close to home, preferably by walking, cycling, and public transport rather than by private car. Moreno et al. (2021) frame the concept around proximity, diversity, density, and digitalisation, but they also warn that it is not simply a radius drawn around a home. For empirical work, the concept must be translated into measurable accessibility: which destinations count, what travel modes are acceptable, and how network travel time is estimated. Pozoukidou and Chatziyiannaki (2021) show that measurement choices strongly shape whether a neighbourhood appears to be a ?complete? neighbourhood. A grid-based method, like the 500 m cells used here, is useful because it avoids relying only on administrative boundaries and allows comparable measurement across central and peripheral districts. However, grid analysis should still be interpreted as a model of proximity, not as a full representation of residents? lived mobility.

Walkability and cycling accessibility research provides the methodological backbone for this project. Ewing and Cervero?s (2010) review of the built environment and travel behaviour highlights density, diversity, design, destination accessibility, and distance to transit as recurring predictors of non-car travel. Sallis et al. (2016), using evidence from multiple international cities, connect neighbourhood environment features with physical activity, reinforcing the idea that nearby destinations and connected street networks can support healthier daily routines. For Track A, this matters because sport and health are not only about formal gyms: parks, everyday food access, cycling infrastructure, and transit-supported access to facilities can all influence whether an active lifestyle is practical.

The equity critique is equally important. The 15-minute city can sound universally beneficial, but access to high-quality amenities is often uneven and may be capitalised into housing prices. Guzman, Oviedo, and Rivera (2017) argue that accessibility analysis should examine who benefits from transport and amenity improvements, not only where facilities are located. Willberg et al. (2023) similarly show that a ?15-minute? label can conceal differences between population groups, travel modes, and service quality. In Shanghai, this critique is central: inner districts may offer dense sport facilities, metro access, and greener public spaces, but also higher housing costs. Peripheral districts may have space for parks and sports fields but weaker walkable access or transit connectivity. This project therefore includes rent band or housing-price proxy fields in the web app, not as a Track C affordability analysis, but as context for interpreting health-lifestyle accessibility.

Health-oriented urban amenity research also supports a broad indicator design. Giles-Corti et al. (2016) argue that city planning is a public-health intervention because street networks, green space, food environments, and transport options shape everyday exposure and activity. Frank et al. (2007) link walkable neighbourhood design with physical activity and body weight outcomes, showing why proximity to exercise-supportive environments can matter beyond recreational preference. For Shanghai Track A, the score therefore combines formal sport facilities such as gyms, swimming pools, courts, and studios with green infrastructure, cycling-lane access, fresh markets, and optional air quality. The approach remains transparent: when true NDVI or AQI cannot be acquired reproducibly, the notebooks either use a documented green-coverage proxy or provide a clearly labelled sample interface excluded from official scoring.

This notebook operationalises those ideas through a reproducible data catalogue. Every dataset is logged with source, collection date, fields, cleaning steps, and limitations. The goal is not to claim perfect real-time truth about Shanghai?s amenities, but to produce a defensible, inspectable workflow from raw spatial data to public H3 web map.

References:

- Moreno, C., Allam, Z., Chabaud, D., Gall, C., & Pratlong, F. (2021). Introducing the ?15-Minute City?: Sustainability, resilience and place identity in future post-pandemic cities. *Smart Cities*, 4(1), 93-111.
- Pozoukidou, G., & Chatziyiannaki, Z. (2021). 15-minute city: Decomposing the new urban planning eutopia. *Sustainability*, 13(2), 928.
- Ewing, R., & Cervero, R. (2010). Travel and the built environment: A meta-analysis. *Journal of the American Planning Association*, 76(3), 265-294.
- Sallis, J. F., Cerin, E., Conway, T. L., Adams, M. A., Frank, L. D., Pratt, M., et al. (2016). Physical activity in relation to urban environments in 14 cities worldwide. *The Lancet*, 387(10034), 2207-2217.
- Guzman, L. A., Oviedo, D., & Rivera, C. (2017). Assessing equity in transport accessibility to work and study: The Bogot? region. *Journal of Transport Geography*, 58, 236-246.
- Willberg, E., Fink, C., & Toivonen, T. (2023). The 15-minute city for all? Measuring individual and temporal variations in walking accessibility. *Journal of Transport Geography*, 106, 103521.
- Giles-Corti, B., Vernez-Moudon, A., Reis, R., Turrell, G., Dannenberg, A. L., Badland, H., et al. (2016). City planning and population health: A global challenge. *The Lancet*, 388(10062), 2912-2924.
- Frank, L. D., Saelens, B. E., Powell, K. E., & Chapman, J. E. (2007). Stepping towards causation: Do built environments or neighborhood and travel preferences explain physical activity, driving, and obesity? *Social Science & Medicine*, 65(9), 1898-1914.

## 0. Setup
Load project utilities, configuration, and optional environment variables. The notebooks are designed to run from the repository root or from the `notebooks/` folder.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from fifteenmc.paths import load_project_paths, load_yaml
from fifteenmc.catalog import build_data_catalog, write_catalog_markdown
from fifteenmc.cleaning import build_processed_datasets, normalize_sports_poi, normalize_transit

paths = load_project_paths(PROJECT_ROOT)
paths

## 1. Data Source Register
The course requires at least four independent datasets. This project uses boundary, road, sports POI, transit, green/land-use, housing, Mobike, and optional AQI datasets. The catalogue below records source, collection date, field notes, cleaning actions, and limitations.

In [ ]:
catalog = build_data_catalog(paths)
catalog

In [ ]:
paths.output('data_catalog').parent.mkdir(parents=True, exist_ok=True)
catalog.to_csv(paths.output('data_catalog'), index=False, encoding='utf-8-sig')
write_catalog_markdown(catalog, PROJECT_ROOT / 'docs' / 'DATA_DICTIONARY.md')
print(paths.output('data_catalog'))
print(PROJECT_ROOT / 'docs' / 'DATA_DICTIONARY.md')

## 2. Raw Data Availability Check
A reproducible pipeline should fail loudly when an expected local file is missing. Optional API-based sources are documented but not required unless credentials are supplied.

In [ ]:
catalog[['dataset_id', 'title', 'collection_date', 'exists', 'local_path']]

## 3. Clean Sports, Fitness, and Transit Datasets
The sports POI layer is mapped to Track A categories. Transit stops and stations are merged into a common accessibility layer. Raw records are not published; cleaned derived files are cached as GeoParquet.

In [ ]:
processed_paths = build_processed_datasets(paths, smoke=False)
processed_paths

In [ ]:
import geopandas as gpd
amenities = gpd.read_parquet(paths.output('amenities'))
transit = gpd.read_parquet(paths.output('transit'))
print('amenities', amenities.shape)
print('transit', transit.shape)
amenities.head()

## 4. Category Validation
This quick check confirms that Track A categories are present after normalization. If a source schema changes, update `config/category_mapping.yaml`.

In [ ]:
amenities['category_group'].value_counts().head(20)

## 5. Data Limitations and Academic Integrity
No API key is embedded in code. If AQI or Gaode routing is used, credentials must be placed in `.env` and outputs should cite the collection date. AI assistance is documented separately in `docs/AI_USE.md`.